In [ ]:
# Step 1: Upload your finetune.jsonl
from google.colab import files
uploaded = files.upload()
# Select your finetune.jsonl file

# Step 2: Install dependencies
!pip install unsloth trl datasets accelerate

# Step 3: Run fine-tuning
import json
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

def load_jsonl(path):
    records = []
    with open(path, "r") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def format_record(record, persona):
    instruction = record.get("instruction", f"You are {persona}. Respond in character.")
    user_input = record.get("input", "")
    output = record.get("output", "")
    if user_input:
        return f"### Instruction:\n{instruction}\n\n### Input:\n{user_input}\n\n### Response:\n{output}"
    return f"### Instruction:\n{instruction}\n\n### Response:\n{output}"

# Load your data
records = load_jsonl("finetune.jsonl")
print(f"Loaded {len(records)} records")

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/phi-2",
    max_seq_length=1024,
    dtype=torch.float32,
    load_in_4bit=True,
)

# Add LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
    use_gradient_checkpointing="unsloth",
)

# Format data
formatted = [{"text": format_record(r, "Steve Jobs")} for r in records]

# Train
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=Dataset.from_list(formatted),
    args=TrainingArguments(
        output_dir="./steve_jobs_model",
        per_device_train_batch_size=2,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=False,
        bf16=False,
        logging_steps=10,
        save_steps=50,
        report_to="none",
    ),
    max_seq_length=1024,
    dataset_text_field="text",
)

print("\n🚀 Starting training...")
trainer.train()

# Save and download
model.save_pretrained("steve_jobs_model")
tokenizer.save_pretrained("steve_jobs_model")

# Zip and download
!zip -r steve_jobs_model.zip steve_jobs_model
files.download("steve_jobs_model.zip")
print("\n✅ Training complete! Model downloaded.")